In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import PatchCollection
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import os
from pathlib import Path
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from minisom import MiniSom
# Global Session Timestamp for Folder Naming
SESSION_TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [2]:
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

In [3]:
# ------------------------- Konfiguration -------------------------
# MANUAL für manuelle Feature-Auswahl
# LOOP für automatische Feature-Kombination
import os
EXECUTION_MODE = os.environ.get('SOM_MODE', 'MANUAL')


# ------------------------- Attribut-Auswahl (verpflichtend) -------------------------
# Die Hauptionen sollten immer dabei sein (FIXED_BASE_FEATURES)

FIXED_BASE_FEATURES = [
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
]


# ------------------------- Attribut-Auswahl (manuell) ------------------------- 
# Alle hier einkommentierten Features werden ZUSÄTZLICH zur Basis genutzt (OPTIONAL_FEATURES_POOL).

OPTIONAL_FEATURES_POOL = [
    # --------- Physikalische Parameter -----------
    "temperature_in_c",
    "pH",
    "electrical_conductivity_25c_in_uS/cm",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",

    # --------- Weitere Ionen / Spurenelemente ---------
    "K_in_mmol/L",
    "NO3_in_mmol/L",
    "Li_in_mmol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "HS_in_mmol/L",
    "O2_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "F_in_umol/L",
    "H2SiO3_in_umol/L",
    
    # --------- Metadaten (nur numerisch nutzen) ---------
    # "depth_bgl_in_m",
    # "stratigraphic_period",
    # "rock_type"
]


print(f"Modus: {EXECUTION_MODE}")
print(f"Fixed Base ({len(FIXED_BASE_FEATURES)}): {FIXED_BASE_FEATURES}")
print(f"Optional Pool ({len(OPTIONAL_FEATURES_POOL)}): {OPTIONAL_FEATURES_POOL}")

Modus: LOOP
Fixed Base (6): ['Na_in_mmol/L', 'Mg_in_mmol/L', 'Ca_in_mmol/L', 'Cl_in_mmol/L', 'SO4_in_mmol/L', 'HCO3_in_mmol/L']
Optional Pool (16): ['temperature_in_c', 'pH', 'electrical_conductivity_25c_in_uS/cm', 'redox_potential_in_mV', 'total_dissolved_solids_in_mmol/L', 'K_in_mmol/L', 'NO3_in_mmol/L', 'Li_in_mmol/L', 'Fe_in_mmol/L', 'Mn_in_mmol/L', 'HS_in_mmol/L', 'O2_in_mmol/L', 'Sr_in_umol/L', 'NH4_in_umol/L', 'F_in_umol/L', 'H2SiO3_in_umol/L']


In [4]:
# ------------------------- Basisverzeichnis (aktuelles Notebook-Verzeichnis) -------------------------
base_dir = Path.cwd()


# ------------------------- Suche nach dem neuesten Preprocessing-Ordner -------------------------
preprocessing_root = base_dir.parent.parent / "3.1_Preprocessing" / "Preprocessing"
timestamp_folders = [f for f in preprocessing_root.iterdir() if f.is_dir()]
if not timestamp_folders:
    raise FileNotFoundError(f"Keine Preprocessing-Ordner in {preprocessing_root} gefunden.")

latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)
print(f"Verwendeter Preprocessing-Ordner: {latest_folder.name}")


# ------------------------- Zeitstempel finden (neuesten Ordner) -------------------------
try:
    prep_ts = datetime.strptime(latest_folder.name, "%Y-%m-%d_%H-%M-%S")
except ValueError:
    print("Warnung: Preprocessing-Ordnername entspricht nicht dem Schema. Nutze Dateisystem-Zeit.")
    prep_ts = datetime.fromtimestamp(latest_folder.stat().st_mtime)


# ------------------------- Lade Preprocessed Data -------------------------
input_path_prep = latest_folder / "Preprocessed_SOM_Ready.csv"
df_full = pd.read_csv(input_path_prep, low_memory=False)

print(f"Daten erfolgreich aus 3.1_Preprocessing geladen! Shape: {df_full.shape}")

Verwendeter Preprocessing-Ordner: 2026-01-08_20-28-21


Daten erfolgreich aus 3.1_Preprocessing geladen! Shape: (94264, 50)


<div class="alert alert-info">
    <b>Temperature Analysis & Hexagonal SOM</b><br><br>
    <b>Datenquelle:</b><br>
    - Preprocessed Data: Ionen (Log + Gaussian Scaling) + Temperatur (Cleaned)<br>
    <br>
    <b>Filter:</b><br>
    - <b>Nur Temperaturen > 10 °C</b> werden berücksichtigt.
    <br>
    <b>Ziel:</b><br>
    - Clustering mittels SOM (Hexagonal).
    - Analyse der Temperaturverteilung innerhalb der Cluster.
    - Export als PDF Report.
</div>

In [5]:
# ------------------ Features-Normierung aus 3.1 ------------------

def get_training_features(user_selection, df_columns):
    training_features = []
    mapping_report = []
    
    for user_col in user_selection:
        found = False
        # --------------- Suche nach transformierter Version (_gauss) --------------------
        candidates = [c for c in df_columns if c.startswith(user_col) and "_gauss" in c]
        if candidates:
            # --------------------- Priorisiere _log_gauss ---------------------
            best_match = next((c for c in candidates if "_log_gauss" in c), candidates[0])
            training_features.append(best_match)
            mapping_report.append(f"{user_col} -> {best_match}")
            found = True
        else:
            # --------------------- Exakter Match ---------------------
            if user_col in df_columns:
                training_features.append(user_col)
                mapping_report.append(f"{user_col} -> {user_col} (Raw)")
                found = True
        
        if not found:
             # --------------------- Fallback ---------------------
             fuzzy = [c for c in df_columns if c.startswith(user_col) and "_gauss" in c]
             if fuzzy:
                 best_match = fuzzy[0]
                 training_features.append(best_match)
                 mapping_report.append(f"{user_col} -> {best_match} (Fuzzy)")
                 found = True
                 
        if not found:
             print(f"Feature '{user_col}' nicht gefunden! Wird ignoriert.")
             
    return training_features, mapping_report

In [6]:
# ------------------ Plotting -----------------
def plot_hex_map_to_axis(ax, data_matrix, title, cmap='viridis', label_suffix='', annotate=False):
    sy, sx = data_matrix.shape
    ax.set_aspect('equal')
    patches = []
    colors = []
    for y in range(sy):
        for x in range(sx):
            val = data_matrix[y, x]
            if pd.isna(val): continue
            offset = 0.5 if y % 2 != 0 else 0.0
            center_x = x + offset
            center_y = y * (np.sqrt(3) / 2)
            radius = 1 / np.sqrt(3)
            hex_poly = mpatches.RegularPolygon((center_x, center_y), numVertices=6, radius=radius, orientation=np.radians(30), edgecolor='k', linewidth=0.5)
            patches.append(hex_poly)
            colors.append(val)
            if annotate:
                ax.text(center_x, center_y, f"{val:.1f}", ha='center', va='center', fontsize=7, fontweight='bold')
    if not patches: return
    collection = PatchCollection(patches, cmap=cmap, alpha=0.9)
    collection.set_array(np.array(colors))
    ax.add_collection(collection)
    ax.set_xlim(-0.5, sx + 0.5)
    ax.set_ylim(-0.5, sy * (np.sqrt(3)/2) + 0.5)
    ax.axis('off')
    ax.set_title(title, fontsize=12)
    return collection

In [7]:
def run_som_analysis(selection_names, run_id_suffix=""):

    # ----------------------- Mapping -----------------------
    train_cols, report = get_training_features(selection_names, df_full.columns)
    print(f"\n--- Start Run: {run_id_suffix} ---")
    print("Mapping:", report)
    if not train_cols:
        print("Keine validen Features. Skip.")
        return
        
    # ----------------------- Filter Data -----------------------
    analysis_cols = [c for c in selection_names if c in df_full.columns]
    mandatory = list(set(train_cols + analysis_cols + ['temperature_in_c', 'rock_type']))
    mandatory = [c for c in mandatory if c in df_full.columns]
    
    df_run = df_full[mandatory].copy()
    
    # ----------------------- Temperaturfilter >= 10°C -----------------------
    if 'temperature_in_c' in df_run.columns:
         df_run['temperature_in_c'] = pd.to_numeric(df_run['temperature_in_c'], errors='coerce')
         cond = (df_run['temperature_in_c'] >= 10) | (df_run['temperature_in_c'].isna())
         df_run = df_run[cond]
    
    # ----------------------- Filter für komplette Daten -----------------------
    df_run.dropna(subset=train_cols, inplace=True)
    if len(df_run) < 50:
        print(f"Zu wenig Daten ({len(df_run)}). Skip.")
        return
        
    # ----------------------- Skalierung -----------------------

    scaler = MinMaxScaler(feature_range=(0, 1))
    data_scaled = scaler.fit_transform(df_run[train_cols].values)
    
    # ----------------------- SOM trainieren -----------------------
    som_x, som_y = 6, 6 # Dimension des SOMs
    som = MiniSom(x=som_x, y=som_y, input_len=len(train_cols), sigma=0.5, learning_rate=0.5, 
                  topology='hexagonal', neighborhood_function='gaussian', activation_distance='euclidean', random_seed=42)
    som.random_weights_init(data_scaled)
    som.train_random(data_scaled, 5000)
    
    #  ---------------------- Analyse ----------------------
    winner_coords = [som.winner(x) for x in data_scaled]
    df_run['som_x'] = [c[0] for c in winner_coords]
    df_run['som_y'] = [c[1] for c in winner_coords]
    
    # ---------------------- Temperatur Karte ----------------------
    temp_map = np.full((som_x, som_y), np.nan)
    if 'temperature_in_c' in df_run.columns:
         agg = df_run.groupby(['som_x', 'som_y'])['temperature_in_c'].mean()
         for (gx, gy), val in agg.items(): temp_map[gx, gy] = val
         
    # -----------------------------------------------------------------------------------------------------

    # ---------------------- PDF exportieren ----------------------
    time_str = datetime.now().strftime("%H-%M-%S")
    
    clean_names = [n.split('_in_')[0] for n in selection_names]
    combo_str = "-".join(clean_names)[:50]
    
    out_dir = base_dir / "MiniSom_Results" / SESSION_TIMESTAMP
    out_dir.mkdir(parents=True, exist_ok=True)
    pdf_path = out_dir / f"Report_{run_id_suffix}_{combo_str}.pdf"
    
    with PdfPages(pdf_path) as pdf:

        plt.figure(figsize=(10,6))
        plt.text(0.5, 0.5, f"Run: {run_id_suffix}\nFeatures: {', '.join(clean_names)}\nData Points: {len(df_run)}", ha='center', fontsize=12)
        plt.axis('off')
        pdf.savefig()
        plt.close()
        
        # ---------------------- Temperatur Karte ----------------------
        if 'temperature_in_c' in df_run.columns:
             f, ax = plt.subplots(figsize=(6,6))
             col_t = plot_hex_map_to_axis(ax, temp_map, "Mean Temp", annotate=True, cmap='coolwarm')
             plt.colorbar(col_t, ax=ax)
             pdf.savefig(f)
             plt.close(f)
             
    print(f"Report saved: {pdf_path}")

In [8]:

# --- Merged Block from Cell d7e5c6af ---
import itertools

# ------------------------------- MANUELLE AUSFÜHRUNG -------------------------------

if EXECUTION_MODE == 'MANUAL':

    print("\n=== Modus: MANUAL ===")

    # ---------------------- Kombiniere Basis + Alle Optionalen ----------------------
    full_selection = FIXED_BASE_FEATURES + OPTIONAL_FEATURES_POOL
    # ---------------------- Duplikate entfernen ----------------------
    full_selection = list(set(full_selection))
    
    print(f"Starte Analyse für {len(full_selection)} Features...")
    run_som_analysis(full_selection, "Manual_Run")
    

# --- Merged Block from Cell 219eaca3 ---
# ------------------------------- LOOP AUSFÜHRUNG -------------------------------

elif EXECUTION_MODE == 'LOOP':

    print("\n=== Modus: LOOP ===")
    
    base = FIXED_BASE_FEATURES
    pool = OPTIONAL_FEATURES_POOL
    
    # ----------------- nur mit Basis als Referenz -----------------
    combos_to_run = [base]
    labels = ["Base_Only"]
    
    # ----------------- alle Kombinationen ----------------
    max_r = len(pool)
    if max_r > 8:
        print("Zu viele optionale Features für vollen Loop. Beschränke auf max 3 Zusatz-Attribute.")
        max_r = 3
        
    for r in range(1, max_r + 1):
        for subset in itertools.combinations(pool, r):
            # ----------------- Neue Features = Basis + Subset -----------------
            combined = list(base) + list(subset)
            combos_to_run.append(combined)
            
            # ----------------- Label für Ordnernamen -----------------
            addon_names = [n.split('_in_')[0] for n in subset]
            labels.append("Plus_" + "-".join(addon_names))

    print(f"{len(combos_to_run)} Runs geplant.")
    
    for i, combination in enumerate(combos_to_run):
        run_label = labels[i]
        # ----------------- ID generieren -----------------
        run_id = f"{i+1:03d}_{run_label}"
        
        print(f"\n--- Loop Schritt {i+1}/{len(combos_to_run)}: {run_label} ---")
        run_som_analysis(combination, run_id)

# --- Merged Block from Cell de57b9ba ---
else:
    print(f"Unbekannter Modus: {EXECUTION_MODE}")



=== Modus: LOOP ===
Zu viele optionale Features für vollen Loop. Beschränke auf max 3 Zusatz-Attribute.
697 Runs geplant.

--- Loop Schritt 1/697: Base_Only ---

--- Start Run: 001_Base_Only ---
Mapping: ['Na_in_mmol/L -> Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L -> Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L -> Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L -> Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L -> SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L -> HCO3_in_mmol/L_log_gauss']


Report saved: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\2026-01-08_20-29-09\Report_001_Base_Only_Na-Mg-Ca-Cl-SO4-HCO3.pdf

--- Loop Schritt 2/697: Plus_temperature ---

--- Start Run: 002_Plus_temperature ---
Mapping: ['Na_in_mmol/L -> Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L -> Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L -> Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L -> Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L -> SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L -> HCO3_in_mmol/L_log_gauss', 'temperature_in_c -> temperature_in_c_log_gauss']


Report saved: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\2026-01-08_20-29-09\Report_002_Plus_temperature_Na-Mg-Ca-Cl-SO4-HCO3-temperature.pdf

--- Loop Schritt 3/697: Plus_pH ---

--- Start Run: 003_Plus_pH ---
Mapping: ['Na_in_mmol/L -> Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L -> Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L -> Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L -> Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L -> SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L -> HCO3_in_mmol/L_log_gauss', 'pH -> pH_gauss']


Report saved: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\2026-01-08_20-29-09\Report_003_Plus_pH_Na-Mg-Ca-Cl-SO4-HCO3-pH.pdf

--- Loop Schritt 4/697: Plus_electrical_conductivity_25c ---

--- Start Run: 004_Plus_electrical_conductivity_25c ---
Mapping: ['Na_in_mmol/L -> Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L -> Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L -> Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L -> Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L -> SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L -> HCO3_in_mmol/L_log_gauss', 'electrical_conductivity_25c_in_uS/cm -> electrical_conductivity_25c_in_uS/cm_log_gauss']


Report saved: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\2026-01-08_20-29-09\Report_004_Plus_electrical_conductivity_25c_Na-Mg-Ca-Cl-SO4-HCO3-electrical_conductivity_25c.pdf

--- Loop Schritt 5/697: Plus_redox_potential ---

--- Start Run: 005_Plus_redox_potential ---
Mapping: ['Na_in_mmol/L -> Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L -> Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L -> Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L -> Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L -> SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L -> HCO3_in_mmol/L_log_gauss', 'redox_potential_in_mV -> redox_potential_in_mV_gauss']


...

Report saved: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\2026-01-08_20-29-09\Report_692_Plus_O2-NH4-H2SiO3_Na-Mg-Ca-Cl-SO4-HCO3-O2-NH4-H2SiO3.pdf

--- Loop Schritt 693/697: Plus_O2-F-H2SiO3 ---

--- Start Run: 693_Plus_O2-F-H2SiO3 ---
Mapping: ['Na_in_mmol/L -> Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L -> Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L -> Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L -> Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L -> SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L -> HCO3_in_mmol/L_log_gauss', 'O2_in_mmol/L -> O2_in_mmol/L_log_gauss', 'F_in_umol/L -> F_in_umol/L_log_gauss', 'H2SiO3_in_umol/L -> H2SiO3_in_umol/L_log_gauss']
Zu wenig Daten (49). Skip.

--- Loop Schritt 694/697: Plus_Sr-NH4-F ---

--- Start Run: 694_Plus_Sr-NH4-F ---
Mapping: ['Na_in_mmol/L -> Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L -> Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L -> Ca_in_mmol/L_log_gaus

Report saved: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\2026-01-08_20-29-09\Report_694_Plus_Sr-NH4-F_Na-Mg-Ca-Cl-SO4-HCO3-Sr-NH4-F.pdf

--- Loop Schritt 695/697: Plus_Sr-NH4-H2SiO3 ---

--- Start Run: 695_Plus_Sr-NH4-H2SiO3 ---
Mapping: ['Na_in_mmol/L -> Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L -> Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L -> Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L -> Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L -> SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L -> HCO3_in_mmol/L_log_gauss', 'Sr_in_umol/L -> Sr_in_umol/L_log_gauss', 'NH4_in_umol/L -> NH4_in_umol/L_log_gauss', 'H2SiO3_in_umol/L -> H2SiO3_in_umol/L_log_gauss']


Report saved: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\2026-01-08_20-29-09\Report_695_Plus_Sr-NH4-H2SiO3_Na-Mg-Ca-Cl-SO4-HCO3-Sr-NH4-H2SiO3.pdf

--- Loop Schritt 696/697: Plus_Sr-F-H2SiO3 ---

--- Start Run: 696_Plus_Sr-F-H2SiO3 ---
Mapping: ['Na_in_mmol/L -> Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L -> Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L -> Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L -> Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L -> SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L -> HCO3_in_mmol/L_log_gauss', 'Sr_in_umol/L -> Sr_in_umol/L_log_gauss', 'F_in_umol/L -> F_in_umol/L_log_gauss', 'H2SiO3_in_umol/L -> H2SiO3_in_umol/L_log_gauss']


Report saved: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\2026-01-08_20-29-09\Report_696_Plus_Sr-F-H2SiO3_Na-Mg-Ca-Cl-SO4-HCO3-Sr-F-H2SiO3.pdf

--- Loop Schritt 697/697: Plus_NH4-F-H2SiO3 ---

--- Start Run: 697_Plus_NH4-F-H2SiO3 ---
Mapping: ['Na_in_mmol/L -> Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L -> Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L -> Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L -> Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L -> SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L -> HCO3_in_mmol/L_log_gauss', 'NH4_in_umol/L -> NH4_in_umol/L_log_gauss', 'F_in_umol/L -> F_in_umol/L_log_gauss', 'H2SiO3_in_umol/L -> H2SiO3_in_umol/L_log_gauss']


Report saved: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\2026-01-08_20-29-09\Report_697_Plus_NH4-F-H2SiO3_Na-Mg-Ca-Cl-SO4-HCO3-NH4-F-H2SiO3.pdf
